<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #4f46e5; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Búsqueda Informada: Greedy Best-First Search (GBFS) 🎯
      </h1>
      <p style="margin: 6px 0 0 0; color: #4f46e5; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #4f46e5; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 04 • Algoritmos de Búsqueda
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #4f46e5; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

---
## 1. De la Búsqueda a Ciegas a la Búsqueda con Pistas (1.16.7) 💡

DFS y BFS (cuaderno 01) exploran el árbol **sin importar el nombre** del archivo que buscan: para ellos, buscar `informe_final.pdf` o buscar `nomina_septiembre.xlsx` produce exactamente el mismo orden de exploración. Eso es lo que las hace *no informadas*.

**Greedy Best-First Search (GBFS)** —Búsqueda Voraz Primero el Mejor— incorpora una **función heurística** `h(n)` que estima, para cada nodo `n`, qué tan "cerca" parece estar del objetivo. En cada paso, GBFS expande **siempre** el nodo de la frontera con el **menor** valor de `h(n)`, sin importar cuánto costó llegar hasta él.

$$\text{GBFS expande el nodo } n \text{ que minimiza } h(n)$$

Es un algoritmo **voraz** (*greedy*) en el sentido estricto: toma en cada paso la decisión que luce mejor *en ese instante*, sin ninguna capacidad de "arrepentirse" del costo ya invertido.

---
## 2. Construyendo una Heurística: Similitud de Nombres 🧮

¿Qué heurística tiene sentido para "qué tan cerca está esta carpeta del archivo que busco"? Una opción razonable —y muy humana— es la **similitud textual** entre el nombre del nodo actual y el nombre del archivo objetivo: si un nodo se llama `Informes`, probablemente esté más relacionado con `informe_final.pdf` que uno llamado `Nomina`.

Usamos `difflib.SequenceMatcher` de la librería estándar de Python, y definimos:

$$h(n) = 1 - \text{similitud}(\text{nombre}(n), \text{objetivo})$$

Así, `h(n) = 0` significa coincidencia perfecta (el nodo *es* el objetivo) y valores cercanos a `1` indican nombres muy distintos. Esta heurística vive en [`file_search_gbfs.py`](file_search_gbfs.py), junto con la implementación de `gbfs_buscar()`.

In [ ]:
import sys
sys.path.append('.')

from file_search import construir_sistema_archivos, imprimir_arbol, dfs_buscar, bfs_buscar, resumen_resultado
from file_search_gbfs import heuristica_similitud_nombre, gbfs_buscar
import inspect

print(inspect.getsource(heuristica_similitud_nombre))

In [ ]:
objetivo = "informe_final.pdf"

# Veamos la heurística en acción para algunos nombres de ejemplo
ejemplos = ["informe_final.pdf", "Informes", "informe_preliminar.pdf",
            "ProyectosIngenieria", "Nomina", "Auditorias"]

print(f"{'Nombre del nodo':<24} | h(n) = 1 - similitud")
print("-" * 50)
for nombre in ejemplos:
    h = heuristica_similitud_nombre(nombre, objetivo)
    print(f"{nombre:<24} | {h}")

Nótese algo importante: `Auditorias` obtiene un valor de `h(n)` relativamente bajo (parece "prometedor") simplemente porque **comparte varias letras sueltas** con `informe_final.pdf`, aunque semánticamente no tenga ninguna relación con el archivo que buscamos. Esta es precisamente la debilidad de una heurística ingenua: mide *parecido superficial de texto*, no *cercanía real en el árbol*.

---
## 3. Implementación de GBFS 🧑‍💻

GBFS organiza la frontera como una **cola de prioridad** (`heapq`), ordenada únicamente por `h(n)`. A diferencia de BFS, **no** hay ninguna noción de "nivel" ni de costo acumulado: sólo importa qué tan prometedor luce cada nodo en este instante.

In [ ]:
print(inspect.getsource(gbfs_buscar))

---
## 4. Ejecutando GBFS sobre el Sistema de Archivos 🏁

In [ ]:
raiz = construir_sistema_archivos()

resultado_gbfs = gbfs_buscar(raiz, objetivo)

print(resumen_resultado("GBFS", resultado_gbfs))
print(f"\nOrden de exploración GBFS ({len(resultado_gbfs['orden_visita'])} nodos):")
print("  " + " → ".join(resultado_gbfs["orden_visita"]))

### 🔬 Análisis del resultado real: ¿por qué GBFS es "voraz"?

Al ejecutar la celda anterior, GBFS expande **12 nodos** en total (frente a los 26 de DFS y 35 de BFS del cuaderno 01) — a primera vista, ¡una gran mejora! Sin embargo, si observamos el **orden de exploración**, el algoritmo no va directo al grano:

```
ServidorCentral → Finanzas → Tesoreria → Contabilidad → balance_general.xlsx
→ Auditorias → auditoria_2024.pdf → auditoria_2025.pdf
→ ProyectosIngenieria → ProyectoNorte → Informes → informe_final.pdf
```

GBFS se **desvió primero hacia la rama de `Finanzas`** (en particular hacia `Auditorias`, con sus archivos `.pdf`), guiado únicamente por la similitud superficial de caracteres con `informe_final.pdf` — ¡ninguno de esos nodos tiene relación real con el objetivo! Sólo después de agotar esa rama poco prometedora, la cola de prioridad finalmente favoreció `ProyectosIngenieria` y el algoritmo llegó al archivo correcto.

Esto ilustra el punto central de esta sección: **GBFS puede ser muy rápido cuando la heurística es buena, pero no ofrece ninguna garantía**. Si en un árbol más grande hubiera más carpetas con nombres "parecidos" al objetivo en ramas equivocadas, GBFS podría llegar a explorar muchísimos más nodos que BFS antes de encontrar el archivo real — o, en un grafo con múltiples caminos, podría incluso terminar encontrando un camino **más largo** de lo necesario, porque nunca considera el costo ya recorrido (`g(n)`).

---
## 5. Comparación de Nodos Expandidos: No Informada vs. GBFS 📊

In [ ]:
resultado_dfs = dfs_buscar(raiz, objetivo)
resultado_bfs = bfs_buscar(raiz, objetivo)

print(f"{'Algoritmo':<8} | {'Nodos expandidos':<18} | {'Usa heurística?'}")
print("-" * 50)
print(f"{'DFS':<8} | {resultado_dfs['nodos_expandidos']:<18} | No")
print(f"{'BFS':<8} | {resultado_bfs['nodos_expandidos']:<18} | No")
print(f"{'GBFS':<8} | {resultado_gbfs['nodos_expandidos']:<18} | Sí (similitud de nombres)")

---
## 6. ¿Por qué GBFS no es Óptimo ni Garantiza Eficiencia? 🚫

| Propiedad | ¿La cumple GBFS? | Explicación |
|---|---|---|
| **Completitud** | ⚠️ Sólo en espacios finitos | En un espacio infinito o con ciclos mal manejados, GBFS puede quedarse "convencido" de seguir una rama poco prometedora indefinidamente. |
| **Optimalidad** | ❌ No garantizada | Al ignorar `g(n)` (el costo ya recorrido), GBFS puede aceptar el primer camino que "luzca bien" según `h(n)`, aunque exista un camino más corto por otra ruta. |
| **Eficiencia** | ⚠️ Depende 100% de la calidad de `h(n)` | Con una heurística excelente, GBFS puede ser el algoritmo más rápido de todos. Con una heurística engañosa (como la nuestra, basada en similitud superficial de texto), puede desperdiciar exploración en callejones sin salida. |

En el próximo cuaderno introduciremos **A\*** (A-estrella), que resuelve exactamente esta debilidad combinando `g(n)` (costo real recorrido) con `h(n)` (estimación heurística), y demostraremos en código por qué eso garantiza encontrar el camino de **menor costo**, no sólo *un* camino.

---
## 📌 Resumen / Puntos Clave

* GBFS es un algoritmo de **búsqueda informada**: usa una función heurística `h(n)` para decidir qué nodo de la frontera expandir a continuación.
* Se implementó una heurística de **similitud de nombres** (`difflib.SequenceMatcher`), que estima —de forma imperfecta— qué tan relacionado está el nombre de un nodo con el del archivo objetivo.
* GBFS organiza su frontera como una **cola de prioridad** ordenada únicamente por `h(n)`, ignorando por completo el costo ya recorrido (`g(n)`).
* En la ejecución real de este cuaderno, GBFS expandió menos nodos que DFS y BFS (12 vs. 26 y 35), pero **se desvió** hacia una rama irrelevante (`Finanzas/Tesoreria/Contabilidad/Auditorias`) guiada por una coincidencia superficial de caracteres.
* GBFS **no garantiza optimalidad**: puede encontrar un camino válido, pero no necesariamente el de menor costo o menor número de saltos.

### 🧠 Autoevaluación

1. ¿Por qué se dice que GBFS es "voraz"? Relaciona tu respuesta con la fórmula que usa para priorizar nodos en la frontera.
2. En la ejecución de este cuaderno, GBFS terminó explorando menos nodos que BFS. ¿Este resultado es una garantía general del algoritmo o depende de la heurística usada? Justifica tu respuesta con lo observado en la Sección 4.
3. Propón una heurística distinta a la similitud de nombres (puede ser tan simple o compleja como quieras) que podría funcionar mejor —o peor— para este problema de búsqueda de archivos. ¿Qué información del dominio usarías?
4. ¿Qué información le falta a GBFS, respecto a BFS, que le impide garantizar el camino de menor costo? ¿Cómo crees que A\* podría resolver ese problema? (Lo verificaremos en el próximo cuaderno.)

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial — Módulo 04: Algoritmos de Búsqueda</i>
  </p>
</div>